# Inspect dotplot export values

Use this notebook to manually inspect the `adata_spatial` object and compare values against the exported dotplot summary CSV.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [ ]:
adata_path = "/home/nnataren/Documents/PhD/Bioinformatics/Banksy_py_fork/Banksy_py/data/xenium/processed/CK_skin_res/adata_spatial_CK_skin_res_0p5.h5ad"
export_csv = "data/xenium/processed/cross_sample_dotplot_exports/local_test_CK_skin_res_0p5_dotplot_summary.csv"
groupby = "labels_scaled_gaussian_pc20_nc0.20_r0.50"

gene = "VWF"
cluster = "0"

In [ ]:
adata = ad.read_h5ad(adata_path)
export = pd.read_csv(export_csv)

adata

## Inspect object structure

In [ ]:
print("adata shape:", adata.shape)
print("adata.raw exists:", adata.raw is not None)
print("groupby exists:", groupby in adata.obs.columns)
print("gene in adata.var_names:", gene in adata.var_names)

if adata.raw is not None:
    print("gene in adata.raw.var_names:", gene in adata.raw.var_names)

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

In [ ]:
adata.obs[groupby].value_counts().sort_index()

## Pull out one gene and one cluster

In [ ]:
def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)

if adata.raw is not None and gene in adata.raw.var_names:
    expr_source = adata.raw
    source_name = "raw"
else:
    expr_source = adata
    source_name = "X"

expr = to_dense(expr_source[:, [gene]].X).ravel()
clusters = adata.obs[groupby].astype(str)
mask = (clusters == str(cluster)).to_numpy()
cluster_expr = expr[mask]

print("expression source:", source_name)
print("gene:", gene)
print("cluster:", cluster)
print("number of cells in cluster:", mask.sum())
print("first 20 expression values:")
cluster_expr[:20]

In [ ]:
manual_mean_expression = cluster_expr.mean()
manual_percent_expressing = (cluster_expr > 0).mean() * 100

manual_mean_expression, manual_percent_expressing

## Compare with export CSV

In [ ]:
export.head()

In [ ]:
export_row = export[
    (export["gene"].astype(str) == gene)
    & (export["cluster_id"].astype(str) == str(cluster))
]

export_row

In [ ]:
if not export_row.empty:
    print("manual mean:", manual_mean_expression)
    print("export mean:", export_row["mean_expression"].iloc[0])
    print("manual percent expressing:", manual_percent_expressing)
    print("export percent expressing:", export_row["percent_expressing"].iloc[0])
    print("manual n_cells:", mask.sum())
    print("export n_cells:", export_row["n_cells"].iloc[0])